### Embedding Models with specific purpose (Code)
> The text embedding models are used mainly for natural language  
> However the code are also text. Since it is not natural language, there is a need for specific models  
> These models can handle code and natural language text in same manner (same vector space)

#### Import of required Libraries

In [1]:
# Sentence transformers to use the embedding models locally
from sentence_transformers import SentenceTransformer, util
import pandas as pd

# Google AI library
from google import genai
from google.genai import types

# Load Environment variables from file
from dotenv import load_dotenv

# Initialise an client object with API key
load_dotenv ()
client = genai.Client()

# Functions from SciPy for check / test
from scipy import spatial

import time

import logging
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)

c:\Users\vaibh\OneDrive\Vaibhav\vEnv\P4\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Utility Functions
> Functions defined for basic comparison and test  
> It can be re-used in different modules

In [2]:
def cosine_similarity(vec1, vec2):

    """
    Function to provide cosine similarity between given 2 vectors.
    Vectors are provided in the form of list of numbers
    Cosine Similarity is returned as a number. 1 being the highest representing high similarity
    """

    # spatial.distance.cosine returns the cosine distance
    # from the distance similarity can be computed, considering the unit vector
    
    return 1 - spatial.distance.cosine(vec1, vec2)

In [3]:
def get_cosine_similarities_HF(model, pairs):
    
    """
    Function to compute similarity scores in each pair for a given list of text pairs.
    Model from HF is given as an argument : as a sentence transformer model.
    For the Text pairs provided, embedding vectors are captured and consine similarities are provided as a list of numbers
    """

    similarities = []
    
    for s1, s2 in pairs:
                
        # Embed the texts
        emb1 = model.encode(s1, convert_to_tensor=True)
        emb2 = model.encode(s2, convert_to_tensor=True)
        
        # Identify the cosine similarity
        sim = util.cos_sim(emb1, emb2).item()
        similarities.append(sim)
    
    return similarities

In [ ]:
# def get_cosine_similarities_gemini(model, pairs):
    
#     """
#     Function to compute similarity scores in each pair for a given list of text pairs.
#     Model from Gemini is given as an argument : Name of the embedding model as string.
#     For the Text pairs provided, embedding vectors are captured and consine similarities are provided as a list of numbers    
#     """

#     similarities = []
    
#     for s1, s2 in pairs:
        
#         result = client.models.embed_content(
#                 model=model,        
#                 contents=[s1, s2],
#                 config=types.EmbedContentConfig(task_type="CODE_RETRIEVAL_QUERY",output_dimensionality=768)
#                 )

#         sim = cosine_similarity (result.embeddings[0].values, result.embeddings[1].values)

#         similarities.append(sim)

#         time.sleep (0.5)

#     return similarities

In [20]:
def get_cosine_similarities_gemini(model, pairs):
    similarities = []

    for s1, s2 in pairs:
        emb1 = client.models.embed_content(
            model=model,
            contents=[s1],
            config=types.EmbedContentConfig(task_type="CODE_RETRIEVAL_QUERY", output_dimensionality=768)
        )
        emb2 = client.models.embed_content(
            model=model,
            contents=[s2],
            config=types.EmbedContentConfig(task_type="CODE_RETRIEVAL_QUERY", output_dimensionality=768)
        )

        sim = cosine_similarity(emb1.embeddings[0].values, emb2.embeddings[0].values)
        similarities.append(sim)

        time.sleep(0.5)

    return similarities

#### Make a Similarity Check  
> Identify Code and corresponding description in natural language  
> Compute vector for the pair and make similarity check  
> Also, pairs of code and incorrect description is made similarity check  
> This Check is done for text embedding model & Code Embedding model (diff models)

In [6]:
# Choose 2 embedders from HF to compare
text = "sentence-transformers/all-MiniLM-L6-v2"
code = "jinaai/jina-code-embeddings-0.5b"
model_text = SentenceTransformer(text)
model_code = SentenceTransformer(code, model_kwargs={"attn_implementation": "eager"})

# Choose an embedding model from Gemini
g = "gemini-embedding-2-preview"

The following layers were not sharded: pooler.dense.weight, embeddings.position_embeddings.weight, embeddings.LayerNorm.bias, encoder.layer.*.attention.self.value.bias, encoder.layer.*.intermediate.dense.weight, encoder.layer.*.attention.self.key.bias, encoder.layer.*.attention.self.query.weight, encoder.layer.*.output.LayerNorm.weight, encoder.layer.*.intermediate.dense.bias, encoder.layer.*.attention.self.query.bias, encoder.layer.*.attention.output.LayerNorm.bias, pooler.dense.bias, embeddings.LayerNorm.weight, encoder.layer.*.attention.output.dense.bias, encoder.layer.*.output.LayerNorm.bias, embeddings.word_embeddings.weight, encoder.layer.*.output.dense.bias, encoder.layer.*.attention.self.value.weight, encoder.layer.*.attention.self.key.weight, encoder.layer.*.output.dense.weight, encoder.layer.*.attention.output.dense.weight, encoder.layer.*.attention.output.LayerNorm.weight, embeddings.token_type_embeddings.weight
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14543.4

> Read Code & Text data from CSV file  
> Make them into correct and incorrect pairs of Code Vs Description

In [7]:
# Load the Test data from CSV file
Similarity_Sample = pd.read_csv ('Code_Embed.csv')

# Text pairs : Code Vs Correct description
Correct_Pairs = list (zip (Similarity_Sample['code'], Similarity_Sample['description']))

# Text pairs : Code Vs Incorrect description
Incorrect_Pairs = list (zip (Similarity_Sample['code'], Similarity_Sample['incorrect_description']))


#### Text Embedding Model

In [8]:
# Use a Text embedding model
# Calculate pair wise similarity For correct and incorrect descriptions
sims_text_corr = get_cosine_similarities_HF(model_text, Correct_Pairs)
sims_text_incorr = get_cosine_similarities_HF(model_text, Incorrect_Pairs)

# Add to the Dataframe
Similarity_Sample['Sim_Text_Corr'] = sims_text_corr
Similarity_Sample['Sim_Text_Incorr'] = sims_text_incorr

Similarity_Sample.to_csv ('Similarities.csv', index=False, float_format="%.3f")

#### Code Embedding Model

In [9]:
# Use a Code embedding model
# Calculate pair wise similarity For correct and incorrect descriptions
sims_code_1_corr = get_cosine_similarities_HF(model_code, Correct_Pairs)
sims_code_1_incorr = get_cosine_similarities_HF(model_code, Incorrect_Pairs)

# Add to the Dataframe
Similarity_Sample['Sim_HF_Code_Corr'] = sims_code_1_corr
Similarity_Sample['Sim_HF_Code_Incorr'] = sims_code_1_incorr

Similarity_Sample.to_csv ('Similarities.csv', index=False, float_format="%.3f")

In [10]:
print(len(Correct_Pairs), len(Incorrect_Pairs))
print(Correct_Pairs[:3], Incorrect_Pairs[:3])

24 24
[('vvcc = 0\nfor vrri in range(len(seq)):\n    vvcc += seq[vrri]\nout = vvcc', 'Returns the cumulative addition of elements from an input collection.'), ('def op_lonohgp(input_data):\n    vbxdvepz = 1\n    for vxfxcjk in range(1, n + 1):\n        vbxdvepz *= vxfxcjk\n    return vbxdvepz', 'Returns n! using a loop-based accumulation.'), ('vcllmwi = []\nfor vmtqriz in data:\n    if vmtqriz % 2 == 0:\n        vcllmwi.append(vmtqriz)\nreturn vcllmwi', 'Extract all even numbers from the input list.')] [('vvcc = 0\nfor vrri in range(len(seq)):\n    vvcc += seq[vrri]\nout = vvcc', 'Check whether the input is a palindrome.'), ('def op_lonohgp(input_data):\n    vbxdvepz = 1\n    for vxfxcjk in range(1, n + 1):\n        vbxdvepz *= vxfxcjk\n    return vbxdvepz', 'Convert all characters to uppercase.'), ('vcllmwi = []\nfor vmtqriz in data:\n    if vmtqriz % 2 == 0:\n        vcllmwi.append(vmtqriz)\nreturn vcllmwi', 'Find the minimum value from the list.')]


#### Gemini Embedding Model with Code task

In [21]:
# Choose an embedding model from Gemini and use it for Code Embedding
# Compute similarity for Correct and Incorrect descriptions
sims_g_Corr = get_cosine_similarities_gemini (g, Correct_Pairs)
sims_g_Incorr = get_cosine_similarities_gemini (g, Incorrect_Pairs)

Similarity_Sample['Sim_G_Code_Corr'] = sims_g_Corr
Similarity_Sample['Sim_G_Code_Incorr'] = sims_g_Incorr

Similarity_Sample.to_csv ('Similarities.csv', index=False, float_format="%.3f")